# Actividad 1 — Análisis de la rotación del inventario por película y tienda.

Métricas principales:

* **Índice de rotación:** cantidad de alquileres / cantidad de copias disponibles.
* **Promedio de días de ociosidad:** cantidad de días entre la devolución de una copia y su siguiente alquiler.

## 1. Configuración del entorno

In [1]:
import sys
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SQL_ACTIVIDAD_1 = PROJECT_ROOT / "sql" / "actividad_1"
sys.path.insert(0, str(PROJECT_ROOT))

from config_conexion import get_engine

## 2. Lectura del conjunto de trabajo

La consulta utilizada para obtener los datos se encuentra en el archivo `sql/actividad_1/conjunto_trabajo.sql`.

Esta consulta devuelve los datos necesarios de inventario y alquileres sin calcular previamente las métricas.

In [2]:
sql_path = SQL_ACTIVIDAD_1 / "conjunto_trabajo.sql"

with open(sql_path, "r", encoding="utf-8") as archivo:
    consulta = archivo.read()

print(consulta)

SELECT
    i.inventario_id AS id,
    i.pelicula_id AS pelicula,
    i.tienda_id AS tienda,
    a.alquiler_id AS alquiler,
    a.fecha_alquiler AS fecha,
    a.fecha_devolucion AS devolucion
FROM inventario AS i
LEFT JOIN alquiler AS a
    ON i.inventario_id = a.inventario_id;


## 3. Carga de datos desde SQL Server

In [3]:
engine = get_engine()

with engine.connect() as conexion:
    df = pd.read_sql(consulta, conexion)

df.head()

,id,pelicula,tienda,alquiler,fecha,devolucion
0,367,80,1,1.0,2005-05-24 22:53:30,2005-05-26 22:04:30
1,1525,333,2,2.0,2005-05-24 22:54:33,2005-05-28 19:40:33
2,1711,373,2,3.0,2005-05-24 23:03:39,2005-06-01 22:12:39
3,2452,535,1,4.0,2005-05-24 23:04:41,2005-06-03 01:43:41
4,2079,450,2,5.0,2005-05-24 23:05:21,2005-06-02 04:33:21


## 4. Obtención de la devolución anterior

Para calcular los días de ociosidad es necesario conocer cuándo fue devuelta una copia antes de volver a ser alquilada.

Primero se ordenan los registros por `id` de inventario y fecha de alquiler. Luego se agrupan los registros por copia física mediante `groupby("id")`.

Sobre la columna `devolucion` se aplica `shift(1)`, desplazando las devoluciones una posición dentro de cada copia. De esta manera, cada alquiler puede conocer la fecha de devolución del alquiler inmediatamente anterior.

Esta operación es equivalente conceptualmente a utilizar `LAG(fecha_devolucion)` particionado por inventario en SQL Server.

In [4]:
df = df.sort_values(
    by=["id", "fecha"]
).copy()

df["devolucion_anterior"] = (
    df.groupby("id")["devolucion"]
      .shift(1)
)

df[
    ["id", "alquiler", "fecha", "devolucion", "devolucion_anterior"]
].head(10)

,id,alquiler,fecha,devolucion,devolucion_anterior
4860,1,4863.0,2005-07-08 19:03:15,2005-07-11 21:29:15,NaT
11428,1,11433.0,2005-08-02 20:13:10,2005-08-11 21:35:10,2005-07-11 21:29:15
14709,1,14714.0,2005-08-21 21:27:43,2005-08-30 22:26:43,2005-08-11 21:35:10
970,2,972.0,2005-05-30 20:21:07,2005-06-06 00:36:07,NaT
2115,2,2117.0,2005-06-17 20:24:00,2005-06-23 17:45:00,2005-06-06 00:36:07
4184,2,4187.0,2005-07-07 10:41:31,2005-07-11 06:25:31,2005-06-23 17:45:00
9444,2,9449.0,2005-07-30 22:02:34,2005-08-06 02:09:34,2005-07-11 06:25:31
15448,2,15453.0,2005-08-23 01:01:01,2005-08-30 20:08:01,2005-08-06 02:09:34
10121,3,10126.0,2005-07-31 21:36:07,2005-08-03 23:59:07,NaT
15416,3,15421.0,2005-08-22 23:56:37,2005-08-25 18:58:37,2005-08-03 23:59:07


## 5. Cálculo de los días de ociosidad

Los días de ociosidad representan el tiempo durante el cual una copia permaneció disponible entre dos alquileres consecutivos.

Se calculan restando la devolución del alquiler anterior a la fecha del alquiler actual:

**días de ociosidad = fecha del nuevo alquiler - devolución anterior**

El primer alquiler de cada copia no posee una devolución anterior, por lo que su valor de ociosidad queda como `NaN`. Este valor no debe reemplazarse por cero, ya que cero indicaría que la copia fue alquilada nuevamente el mismo día.

In [5]:
# DATEDIFF(DAY, inicio, fin) cuenta cambios de fecha de calendario.
# Se normalizan las horas para mantener la misma definicion en pandas.
df["dias_ociosidad"] = (
    df["fecha"].dt.normalize() - df["devolucion_anterior"].dt.normalize()
).dt.days

df[
    ["id", "pelicula", "tienda", "fecha", "devolucion_anterior", "dias_ociosidad"]
].head(10)

,id,pelicula,tienda,fecha,devolucion_anterior,dias_ociosidad
4860,1,1,1,2005-07-08 19:03:15,NaT,NaN
11428,1,1,1,2005-08-02 20:13:10,2005-07-11 21:29:15,22.0
14709,1,1,1,2005-08-21 21:27:43,2005-08-11 21:35:10,10.0
970,2,1,1,2005-05-30 20:21:07,NaT,NaN
2115,2,1,1,2005-06-17 20:24:00,2005-06-06 00:36:07,11.0
4184,2,1,1,2005-07-07 10:41:31,2005-06-23 17:45:00,14.0
9444,2,1,1,2005-07-30 22:02:34,2005-07-11 06:25:31,19.0
15448,2,1,1,2005-08-23 01:01:01,2005-08-06 02:09:34,17.0
10121,3,1,1,2005-07-31 21:36:07,NaT,NaN
15416,3,1,1,2005-08-22 23:56:37,2005-08-03 23:59:07,19.0


## 6. Promedio de días de ociosidad

Una vez calculada la ociosidad entre alquileres, los registros se agrupan por película y tienda.

Para cada combinación se obtiene la media de `dias_ociosidad` mediante `mean`.

Los valores `NaN`, correspondientes principalmente al primer alquiler de cada copia, no participan del cálculo del promedio (`mean` de pandas los ignora).

In [6]:
ociosidad_promedio = (
    df.groupby(["pelicula", "tienda"], as_index=False)
      .agg(
          promedio_dias_ociosidad=("dias_ociosidad", "mean")
      )
)

ociosidad_promedio.head(10)

,pelicula,tienda,promedio_dias_ociosidad
0,1,1,16.250000
1,1,2,16.250000
2,2,2,59.750000
3,3,2,17.500000
4,4,1,16.444444
5,4,2,44.142857
6,5,2,33.555556
7,6,1,15.125000
8,6,2,13.571429
9,7,1,14.250000


## 7. Cálculo del índice de rotación

El índice de rotación permite relacionar la cantidad de alquileres con la cantidad de copias disponibles de una película en cada tienda.

La fórmula utilizada es:

**índice de rotación = cantidad de alquileres / cantidad de copias**

Para obtener la cantidad de copias se utiliza `nunique` sobre el identificador de inventario. Esto es necesario porque una misma copia puede aparecer varias veces en el conjunto de datos debido a sus distintos alquileres.

Para obtener la cantidad de alquileres se utiliza `count` sobre la columna `alquiler`. Los valores nulos no se cuentan, por lo que las copias que nunca fueron alquiladas permanecen en el inventario pero no incrementan la cantidad de alquileres.

In [7]:
rotacion = (
    df.groupby(["pelicula", "tienda"], as_index=False)
      .agg(
          cantidad_copias=("id", "nunique"),
          cantidad_alquileres=("alquiler", "count")
      )
)

rotacion["indice_rotacion"] = (
    rotacion["cantidad_alquileres"]
    / rotacion["cantidad_copias"]
)

rotacion.head(10)

,pelicula,tienda,cantidad_copias,cantidad_alquileres,indice_rotacion
0,1,1,4,12,3.000000
1,1,2,4,11,2.750000
2,2,2,3,7,2.333333
3,3,2,4,12,3.000000
4,4,1,4,13,3.250000
5,4,2,3,10,3.333333
6,5,2,3,12,4.000000
7,6,1,3,11,3.666667
8,6,2,3,10,3.333333
9,7,1,2,6,3.000000


## 8. Resultado de las métricas calculadas con pandas

Finalmente se combinan los resultados de rotación y ociosidad mediante las columnas `pelicula` y `tienda`.

El DataFrame resultante contiene, para cada película y tienda:

- cantidad de copias;
- cantidad de alquileres;
- índice de rotación;
- promedio de días de ociosidad.

Este resultado podrá compararse posteriormente con las mismas métricas calculadas directamente en T-SQL.

In [8]:
resultado_pandas = rotacion.merge(
    ociosidad_promedio,
    on=["pelicula", "tienda"],
    how="left"
)

resultado_pandas.head(10)

,pelicula,tienda,cantidad_copias,cantidad_alquileres,indice_rotacion,promedio_dias_ociosidad
0,1,1,4,12,3.000000,16.250000
1,1,2,4,11,2.750000,16.250000
2,2,2,3,7,2.333333,59.750000
3,3,2,4,12,3.000000,17.500000
4,4,1,4,13,3.250000,16.444444
5,4,2,3,10,3.333333,44.142857
6,5,2,3,12,4.000000,33.555556
7,6,1,3,11,3.666667,15.125000
8,6,2,3,10,3.333333,13.571429
9,7,1,2,6,3.000000,14.250000


## 9. Ejecución de la métrica en T-SQL

La misma métrica se ejecuta en SQL Server desde `sql/actividad_1/metricas_rotacion.sql`, sin copiar la consulta dentro del notebook. SQL Server usa `LAG` y agrupa por película y tienda. Se mide la consulta completa para informar su tiempo por separado.

In [9]:
from time import perf_counter

consulta_metricas = (SQL_ACTIVIDAD_1 / 'metricas_rotacion.sql').read_text(encoding='utf-8')
tiempos_sql = []
for _ in range(10):
    inicio = perf_counter()
    with engine.connect() as conexion:
        resultado_sql = pd.read_sql(consulta_metricas, conexion)
    tiempos_sql.append(perf_counter() - inicio)

resultado_sql.head() # type: ignore

,pelicula,tienda,cantidad_copias,cantidad_alquileres,indice_rotacion,promedio_dias_ociosidad
0,1,1,4,12,3.000000,16.250000
1,1,2,4,11,2.750000,16.250000
2,2,2,3,7,2.333333,59.750000
3,3,2,4,12,3.000000,17.500000
4,4,1,4,13,3.250000,16.444444


## 10. Comparación entre T-SQL y pandas

Los resultados se unen por `pelicula` y `tienda`. Los conteos se comparan exactamente y las métricas decimales se validan antes de aplicar formato de presentación.

### Hallazgo sobre redondeo

En una comparación inicial se observaron diferencias de 0,01 en algunos promedios de días de ociosidad. La revisión mostró que la fórmula era equivalente, pero ambas implementaciones reducían el promedio a dos decimales antes de compararlo. En valores situados en un punto intermedio, SQL Server y pandas pueden resolver el redondeo de manera diferente.

Por ese motivo, `metricas_rotacion.sql` conserva doce decimales y pandas mantiene los valores sin `.round(2)`. La validación usa `assert_frame_equal` con una tolerancia absoluta de `1e-9`, limitada a la representación numérica de punto flotante. El formato de dos decimales queda reservado para una eventual presentación, no para calcular ni validar.

In [10]:
claves = ["pelicula", "tienda"]

columnas_decimales = [
    "indice_rotacion",
    "promedio_dias_ociosidad"
]

sql_comparacion = (
    resultado_sql
    .sort_values(claves)
    .reset_index(drop=True)
    .copy()
)

pandas_comparacion = (
    resultado_pandas
    .sort_values(claves)
    .reset_index(drop=True)
    .copy()
)

for columna in columnas_decimales:
    sql_comparacion[columna] = pd.to_numeric(
        sql_comparacion[columna],
        errors="raise"
    ).astype(float)

    pandas_comparacion[columna] = pd.to_numeric(
        pandas_comparacion[columna],
        errors="raise"
    ).astype(float)

validacion_equivalente = False

try:
    pd.testing.assert_frame_equal(
        sql_comparacion,
        pandas_comparacion,
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-9
    )

    validacion_equivalente = True
    print("VALIDACIÓN EXITOSA")
    print("-" * 45)
    print(f"Filas comparadas:        {len(sql_comparacion)}")
    print(f"Columnas comparadas:     {len(sql_comparacion.columns)}")
    print(f"Grupos película/tienda:  {len(sql_comparacion)}")
    print(f"Tolerancia numérica:     1e-9")
    print()
    print("Resultados:")
    print("Cantidad de copias coincide")
    print("Cantidad de alquileres coincide")
    print("Índice de rotación coincide")
    print("Promedio de ociosidad coincide")
    print()
    print("Conclusión:")
    print("T-SQL y pandas producen resultados equivalentes.")

except AssertionError as error:
    print("VALIDACIÓN FALLIDA")
    print("-" * 45)
    print("Se encontraron diferencias entre T-SQL y pandas.")
    print()
    print(error)

VALIDACIÓN EXITOSA
---------------------------------------------
Filas comparadas:        1521
Columnas comparadas:     6
Grupos película/tienda:  1521
Tolerancia numérica:     1e-9

Resultados:
Cantidad de copias coincide
Cantidad de alquileres coincide
Índice de rotación coincide
Promedio de ociosidad coincide

Conclusión:
T-SQL y pandas producen resultados equivalentes.


## 11. Medición de tiempos

El tiempo de T-SQL incluye ejecutar la consulta agregada y recibir su resultado desde SQL Server. El de pandas mide solamente ordenar, aplicar `groupby`/`shift` y calcular/agrupar métricas sobre el DataFrame ya cargado. Se hacen tres repeticiones y se informa el promedio; no permiten sacar conclusiones absolutas de rendimiento.

In [11]:
def calcular_metricas_pandas(datos_crudos):
    datos = datos_crudos.sort_values(['id', 'fecha']).copy()
    datos['devolucion_anterior'] = datos.groupby('id')['devolucion'].shift(1)
    datos['dias_ociosidad'] = (datos['fecha'].dt.normalize() - datos['devolucion_anterior'].dt.normalize()).dt.days
    ociosidad = datos.groupby(['pelicula', 'tienda'], as_index=False).agg(promedio_dias_ociosidad=('dias_ociosidad', 'mean'))
    rotacion_temporal = datos.groupby(['pelicula', 'tienda'], as_index=False).agg(cantidad_copias=('id', 'nunique'), cantidad_alquileres=('alquiler', 'count'))
    rotacion_temporal['indice_rotacion'] = rotacion_temporal['cantidad_alquileres'] / rotacion_temporal['cantidad_copias']
    return rotacion_temporal.merge(ociosidad, on=['pelicula', 'tienda'], how='left')

tiempos_pandas = []
for _ in range(10):
    inicio = perf_counter()
    resultado_pandas_temporal = calcular_metricas_pandas(df.drop(columns=['devolucion_anterior', 'dias_ociosidad']))
    tiempos_pandas.append(perf_counter() - inicio)
assert resultado_pandas_temporal.equals(resultado_pandas) # type: ignore
tabla_tiempos = pd.DataFrame({'implementacion': ['T-SQL', 'pandas'], 'promedio_segundos': [np.mean(tiempos_sql), np.mean(tiempos_pandas)], 'minimo_segundos': [np.min(tiempos_sql), np.min(tiempos_pandas)], 'repeticiones': [len(tiempos_sql), len(tiempos_pandas)]})
tabla_tiempos

,implementacion,promedio_segundos,minimo_segundos,repeticiones
0,T-SQL,0.050347,0.048612,10
1,pandas,0.013827,0.012242,10


## 12. Conclusión de la Actividad 1

La implementación en T-SQL y la implementación vectorizada en pandas producen métricas equivalentes para cada combinación de película y tienda, dentro de la tolerancia numérica definida. La diferencia observada en los valores mostrados se explica por el redondeo de presentación, no por una diferencia de cálculo.

En esta medición, pandas resultó más rápido porque calcula sobre un `DataFrame` ya cargado en memoria, mientras que el tiempo de T-SQL incluye la ejecución de la consulta en SQL Server y la transferencia del resultado. Esta comparación no permite afirmar que pandas sea siempre más eficiente: con volúmenes mayores, filtros o agregaciones más costosas, es esperable que SQL Server tenga ventajas al procesar los datos cerca de su almacenamiento.

Por esta razón, se utiliza SQL Server para construir, filtrar y agregar el conjunto de trabajo, y pandas para transformaciones analíticas y validación de resultados.